In [1]:
import pandas as pd
import numpy as np
import re
import glob

In [2]:
df = pd.read_csv("../raw_data/amazon_india_2021.csv")

In [92]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2021_00000001,2021-01-29,CUST_2021_00014071,PROD_000178,Xiaomi Mi 5 64GB Black,Electronics,Smartphones,Xiaomi,43292.78,22.96,...,False,NaN,4.5,Delivered,1,2021,1,0.24,True,3.7
1,TXN_2021_00000002,2021-01-10,CUST_2019_00006495,PROD_000610,Vivo V15 Pro 256GB White,Electronics,Smartphones,Vivo,32964.72,16.68,...,False,NaN,5.0,Delivered,1,2021,1,0.19,True,4.1
2,TXN_2021_00000003,2021-01-18,CUST_2018_00029239,PROD_000404,Xiaomi Poco F1 128GB Gold,Electronics,Smartphones,Xiaomi,45794.75,0.00,...,False,NaN,NaN,Delivered,1,2021,1,0.22,True,4.4
3,TXN_2021_00000004,2021-01-08,CUST_2021_00040991,PROD_000664,Apple iPhone SE (2nd gen) 64GB Gold,Electronics,Smartphones,Apple,136337.08,0.00,...,False,NaN,4.0,Delivered,1,2021,1,0.17,True,3.8
4,TXN_2021_00000005,2021-01-02,CUST_2016_00006478,PROD_000466,Oppo A3s 64GB Black,Electronics,Smartphones,Oppo,19992.12,0.00,...,False,NaN,3.0,Delivered,1,2021,1,0.24,1,3.3


In [93]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(11040), 138187)

In [94]:
df["delivery_charges"].describe()

count    127147.000000
mean          0.000629
std           0.158643
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          40.000000
Name: delivery_charges, dtype: float64

In [95]:
df.shape

(138187, 34)

In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138187 entries, 0 to 138186
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          138187 non-null  object 
 1   order_date              138187 non-null  object 
 2   customer_id             138187 non-null  object 
 3   product_id              138187 non-null  object 
 4   product_name            138187 non-null  object 
 5   category                138187 non-null  object 
 6   subcategory             138187 non-null  object 
 7   brand                   138187 non-null  object 
 8   original_price_inr      138187 non-null  object 
 9   discount_percent        138187 non-null  float64
 10  discounted_price_inr    138187 non-null  float64
 11  quantity                138187 non-null  int64  
 12  subtotal_inr            138187 non-null  float64
 13  delivery_charges        127147 non-null  float64
 14  final_amount_inr    

In [97]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_city', 'customer_state', 'customer_tier',
       'customer_spending_tier', 'customer_age_group', 'payment_method',
       'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale',
       'festival_name', 'customer_rating', 'return_status', 'order_month',
       'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [98]:
df["order_date"].head(20)

0     2021-01-29
1     2021-01-10
2     2021-01-18
3     2021-01-08
4     2021-01-02
5     2021-01-16
6     2021-01-09
7     2021-01-27
8     2021-01-30
9     2021-01-20
10    2021-01-07
11    2021-01-21
12    2021-01-14
13    2021-01-06
14    2021-01-04
15    2021-01-25
16    2021-01-23
17    2021-01-26
18    2021-01-21
19    2021-01-16
Name: order_date, dtype: object

In [99]:
df["order_date"] = (
    df["order_date"]
    .str.replace(" ", "", regex=False)
    .str.replace("/", "-", regex=False)
)

parts = df["order_date"].str.split("-", expand=True)

year_last = parts[2].str.len() == 4

df.loc[year_last, "order_date"] = (
    parts[2] + "-" + parts[0] + "-" + parts[1]
)

parts = df["order_date"].str.split("-", expand=True)

mask = parts[1].astype(int) > 12

df.loc[mask, "order_date"] = (
    parts[0] + "-" + parts[2] + "-" + parts[1]
)

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

In [100]:
df["order_date"].min(), df["order_date"].max()

(Timestamp('2021-01-01 00:00:00'), Timestamp('2021-12-31 00:00:00'))

In [101]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [102]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [103]:
df["original_price_inr"].unique()[:20]

array([ 43292.78,  32964.72,  45794.75, 136337.08,  19992.12,  43813.38,
        92190.83,  57513.72, 147095.42, 161292.12, 212509.  , 159526.67,
        35748.18,  25186.58,  59579.6 , 127446.49,  22441.34, 106984.36,
        39094.74,  41661.86])

In [104]:
mask = df["original_price_inr"].isna()

df.loc[mask, "original_price_inr"] = np.where(
    df.loc[mask, "discount_percent"] == 0,
    
    # Case 1: no discount
    df.loc[mask, "discounted_price_inr"],
    
    # Case 2: discount present
    df.loc[mask, "discounted_price_inr"] / (1 - df.loc[mask, "discount_percent"] / 100)
)

In [105]:
df["original_price_inr"].dtypes

dtype('float64')

In [106]:
df["original_price_inr"].isna().sum()

np.int64(0)

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [107]:
df["customer_rating"] = df["customer_rating"].astype(str)

df["customer_rating"] = df["customer_rating"].str.replace(" stars", "", regex=False)

df["customer_rating"] = df["customer_rating"].str.split("/").str[0]

df["customer_rating"] = pd.to_numeric(df["customer_rating"], errors="coerce")

In [108]:
df["customer_rating"].describe()

count    96302.000000
mean         4.309454
std          0.573539
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [109]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    31603
5.0    24724
4.0    24352
3.5     9797
3.0     5826
Name: count, dtype: int64

In [110]:
df["customer_rating"].isna().sum()

np.int64(41885)

In [111]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    41885
4.5    31603
5.0    24724
4.0    24352
3.5     9797
3.0     5826
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [112]:
df["customer_city"] = df["customer_city"].str.strip().str.lower()

In [113]:
df["customer_city"].unique()

array(['mumbai', 'jaipur', 'bareilly', 'surat', 'vadodara', 'kolkata',
       'bangalore', 'saharanpur', 'coimbatore', 'kanpur', 'chennai',
       'pune', 'lucknow', 'indore', 'delhi', 'ahmedabad', 'kochi',
       'allahabad', 'chandigarh', 'bhubaneswar', 'visakhapatnam',
       'ludhiana', 'nagpur', 'patna', 'hyderabad', 'aligarh', 'varanasi',
       'meerut', 'madras', 'gorakhpur', 'calcutta', 'moradabad', 'mumba',
       'bombay', 'chenai', 'bengaluru', 'banglore', 'bengalore',
       'delhi ncr', 'new delhi'], dtype=object)

In [114]:
city_map = {
    "madras": "chennai",
    "chenai": "chennai",

    "calcutta": "kolkata",

    "bombay": "mumbai",
    "mumba": "mumbai",

    "banglore": "bangalore",
    "bengalore": "bangalore",
    "bengaluru": "bangalore",

    "new delhi": "delhi",
    "delhi ncr": "delhi"
}

In [115]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [116]:
df["customer_city"] = df["customer_city"].str.title()

In [117]:
df["customer_city"].value_counts().head(20)

customer_city
Mumbai           16896
Delhi            14471
Bangalore        12501
Chennai           9977
Pune              8491
Kolkata           7861
Ahmedabad         6623
Hyderabad         5292
Jaipur            5213
Surat             5180
Nagpur            4671
Kanpur            4446
Lucknow           4301
Indore            4224
Coimbatore        3377
Kochi             3086
Visakhapatnam     2821
Patna             2809
Vadodara          2684
Bhubaneswar       2680
Name: count, dtype: int64

In [118]:
df["customer_city"].unique()

array(['Mumbai', 'Jaipur', 'Bareilly', 'Surat', 'Vadodara', 'Kolkata',
       'Bangalore', 'Saharanpur', 'Coimbatore', 'Kanpur', 'Chennai',
       'Pune', 'Lucknow', 'Indore', 'Delhi', 'Ahmedabad', 'Kochi',
       'Allahabad', 'Chandigarh', 'Bhubaneswar', 'Visakhapatnam',
       'Ludhiana', 'Nagpur', 'Patna', 'Hyderabad', 'Aligarh', 'Varanasi',
       'Meerut', 'Gorakhpur', 'Moradabad'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [119]:
bool_candidates = []

bool_values = {"true","false","yes","no","y","n","1","0"}

for col in df.columns:
    vals = set(df[col].astype(str).str.lower().dropna().unique())
    
    if vals & bool_values:
        bool_candidates.append(col)

bool_candidates

['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [120]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [121]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
False            True               False               43833
True             True               False               34853
False            True               True                19264
True             True               True                15428
False            False              False                9350
True             False              False                7851
False            False              True                 4097
True             False              True                 3511
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [3]:
df["category"].value_counts().head(20)

category
Electronics                  138107
ELECTRONICS                      26
Electronics & Accessories        20
Electronic                       19
Electronicss                     15
Name: count, dtype: int64

In [4]:
category_map = {
    "ELECTRONICS": "Electronics",
    "Electronics & Accessories": "Electronics",
    "Electronic": "Electronics",
    "Electronicss": "Electronics"
}

In [5]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [6]:
df["category"].value_counts()

category
Electronics    138187
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [126]:
df["delivery_days"].unique()

array(['1', '5', '2', '4', '3', '6', '1-2 days', '7', 'Same Day', '-1',
       '0', 'Express', '15'], dtype=object)

In [127]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [128]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [129]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [130]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [131]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [132]:
df["delivery_days"].unique()

array([ 1.,  5.,  2.,  4.,  3.,  6.,  7.,  0., nan, 15.])

In [133]:
df["delivery_days"].isnull().sum()

np.int64(874)

In [134]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [135]:
df["delivery_days"].describe()

count    138187.000000
mean          3.154682
std           1.756864
min           0.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          15.000000
Name: delivery_days, dtype: float64

In [136]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [137]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [138]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
251,TXN_2021_00000252,2021-01-31,CUST_2017_00014282,PROD_000487,Apple iPhone 11 256GB Blue,Electronics,Smartphones,Apple,187545.53,20.11,...,False,NaN,5.0,Delivered,1,2021,1,0.21,False,3.5
318,TXN_2021_00000319,2021-01-24,CUST_2021_00000027,PROD_000148,Samsung Galaxy J7 Prime 32GB Black,Electronics,Smartphones,Samsung,29539.19,69.84,...,True,Republic Day Sale,NaN,Delivered,1,2021,1,0.19,True,3.2
1519,TXN_2021_00001520,2021-01-07,CUST_2021_00006407,PROD_001891,Samsung Tracker,Electronics,Smart Watch,Samsung,77317.96,0.00,...,False,NaN,4.5,Delivered,1,2021,1,0.04,False,4.4
1651,TXN_2021_00001652,2021-01-03,CUST_2017_00011764,PROD_000579,Realme Realme 3 128GB Black,Electronics,Smartphones,Realme,32907.49,0.00,...,False,NaN,5.0,Delivered,1,2021,1,0.21,False,4.5
1835,TXN_2021_00001836,2021-01-10,CUST_2021_00030116,PROD_000263,OnePlus OnePlus 5 32GB Black,Electronics,Smartphones,OnePlus,113292.56,0.00,...,False,NaN,NaN,Delivered,1,2021,1,0.25,False,3.3


In [139]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [140]:
duplicates.shape

(1346, 34)

In [141]:
df.duplicated().sum()

np.int64(0)

In [142]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
136724,TXN_2021_00136725,2021-12-22,CUST_2015_00001755,PROD_000553,OnePlus OnePlus 7T 64GB Blue,Electronics,Smartphones,OnePlus,114944.46,0.00,...,False,NaN,NaN,Delivered,12,2021,4,0.19,True,3.7
137957,TXN_2021_00136725_DUP,2021-12-22,CUST_2015_00001755,PROD_000553,OnePlus OnePlus 7T 64GB Blue,Electronics,Smartphones,OnePlus,114944.46,0.00,...,False,NaN,NaN,Delivered,12,2021,4,0.19,True,3.7
13476,TXN_2021_00013477,2021-02-02,CUST_2015_00004799,PROD_000673,Samsung Galaxy S20+ 64GB Black,Electronics,Smartphones,Samsung,82204.14,0.00,...,False,NaN,4.5,Delivered,2,2021,1,0.19,True,4.0
137717,TXN_2021_00013477_DUP,2021-02-02,CUST_2015_00004799,PROD_000673,Samsung Galaxy S20+ 64GB Black,Electronics,Smartphones,Samsung,82204.14,0.00,...,False,NaN,4.5,Delivered,2,2021,1,0.19,True,4.0
135632,TXN_2021_00135633,2021-12-16,CUST_2015_00006062,PROD_001531,HP Gaming 8GB RAM Silver,Electronics,Laptops,HP,135139.93,5.91,...,False,NaN,5.0,Delivered,12,2021,4,1.98,False,4.0
137865,TXN_2021_00135633_DUP,2021-12-16,CUST_2015_00006062,PROD_001531,HP Gaming 8GB RAM Silver,Electronics,Laptops,HP,135139.93,5.91,...,False,NaN,5.0,Delivered,12,2021,4,1.98,False,4.0
94935,TXN_2021_00094936,2021-09-19,CUST_2015_00007436,PROD_000752,Realme Realme Narzo 10 128GB Black,Electronics,Smartphones,Realme,40083.38,0.00,...,False,NaN,5.0,Delivered,9,2021,3,0.19,True,4.3
137685,TXN_2021_00094936_DUP,2021-09-19,CUST_2015_00007436,PROD_000752,Realme Realme Narzo 10 128GB Black,Electronics,Smartphones,Realme,40083.38,0.00,...,False,NaN,5.0,Delivered,9,2021,3,0.19,True,4.3
16937,TXN_2021_00016938,2021-02-07,CUST_2015_00007786,PROD_001892,Samsung Tracker Premium,Electronics,Smart Watch,Samsung,38783.21,0.00,...,False,NaN,3.5,Delivered,2,2021,1,0.06,True,3.5
138076,TXN_2021_00016938_DUP,2021-02-07,CUST_2015_00007786,PROD_001892,Samsung Tracker Premium,Electronics,Smart Watch,Samsung,38783.21,0.00,...,False,NaN,3.5,Delivered,2,2021,1,0.06,True,3.5


In [143]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00001755  PROD_000553  2021-12-22  114944.46             2
CUST_2015_00004799  PROD_000673  2021-02-02  82204.14              2
CUST_2015_00006062  PROD_001531  2021-12-16  135139.93             2
CUST_2015_00007436  PROD_000752  2021-09-19  40083.38              2
CUST_2015_00007786  PROD_001892  2021-02-07  38783.21              2
CUST_2015_00010268  PROD_001535  2021-03-22  183272.23             2
CUST_2015_00010890  PROD_000057  2021-03-08  112473.45             2
CUST_2015_00011923  PROD_000578  2021-07-17  21792.94              2
CUST_2016_00000089  PROD_000402  2021-12-24  34242.15              2
CUST_2016_00000273  PROD_000124  2021-03-24  132288.34             2
dtype: int64

In [144]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [145]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [146]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2017_00014282,PROD_000487,2,2
1,CUST_2021_00000027,PROD_000148,1,2
2,CUST_2021_00006407,PROD_001891,1,2
3,CUST_2017_00011764,PROD_000579,1,2
4,CUST_2021_00030116,PROD_000263,2,2


In [147]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [148]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [149]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [150]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

In [151]:
df[df["transaction_id"]=="TXN_2015_00000280"]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [152]:
# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers ──────────────────────────────────
subcategory_caps = {
    "Smart Watch":        90000,     # Apple Watch Ultra range
    "Tablets":            150000,    # iPad Pro 2021
    "Smartphones":        350000,    # iPhone 13 Pro Max
    "Laptops":            350000,    # High-end gaming laptops
    "TV & Entertainment": 400000,    # 85" QLED
    "Audio":              80000,     # High-end headphones
}
outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: Fill delivery_charges ─────────────────────
df["delivery_charges"] = df["delivery_charges"].fillna(0)

# ── FIX 4: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = (df["subtotal_inr"] + df["delivery_charges"]).round(2)

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory")["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")



Negative prices fixed: 346
Outliers fixed: 1612
                         min        max       mean        50%
subcategory                                                  
Audio                 861.92   74690.60   25367.14   28065.43
Laptops              3689.70  293793.87  121449.10  105880.91
Smart Watch          1971.60   83458.23   49518.48   50465.68
Smartphones          3536.81  349336.00   74964.74   49231.90
TV & Entertainment  13926.60  327877.95  142704.66  139265.97
Tablets              1550.09  141958.76   71705.30   67575.51

NaN in final_amount_inr:   0
Negative prices remaining: 0


In [153]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch: ✅ All within cap

Tablets: ✅ All within cap

Smartphones: ✅ All within cap

Laptops: ✅ All within cap

TV & Entertainment: ✅ All within cap

Audio: ✅ All within cap


Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [154]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
UPI            55116
COD            24977
Credit Card    22061
Debit Card     19467
Net Banking     9718
Wallet          6848
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [155]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [156]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [157]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
75.21829509735107 MB


In [158]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 138187

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        95887       69.39
customer_rating      41885       30.31


In [159]:
df.to_csv("data_cleaning_2021.csv", index=False)
print("File saved successfully!")

File saved successfully!
